In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
from tensorflow.keras.models import load_model

# Only load the model if it is not already available in the notebook
if 'model' not in globals():
    model = load_model('model.h5')

# Load the pickled objects only if they are not already defined
if 'onehot_encoder_geo' not in globals():
    with open('onehot_encoder_geo.pkl', 'rb') as file:
        onehot_encoder_geo = pickle.load(file)

if 'label_encoder_gender' not in globals():
    with open('label_encoder_gender.pkl', 'rb') as file:
        label_encoder_gender = pickle.load(file)

if 'scaler' not in globals():
    with open('scaler.pkl', 'rb') as file:
        scaler = pickle.load(file)

In [3]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
import pandas as pd

# Load your dataset
df = pd.read_csv('Churn_Modelling.csv')

In [5]:
import pickle

with open('onehot_encoder_geo.pkl', 'rb') as file:
    temp_obj = pickle.load(file)

print("Type stored in pkl file:", type(temp_obj))

Type stored in pkl file: <class 'scipy.sparse._csr.csr_matrix'>


In [6]:
# --- IN YOUR TRAINING NOTEBOOK ---

from sklearn.preprocessing import OneHotEncoder
import pickle

# 1. Fit the encoder object
onehot_encoder_geo = OneHotEncoder(sparse_output=False)
onehot_encoder_geo.fit(df[['Geography']])

# 2. Dump the ENCODER OBJECT (not the transformed data)
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

In [7]:
import pandas as pd

# Define your input data dictionary
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

# One-hot encode Geography directly from input_data
geo_encoded_matrix = onehot_encoder_geo.transform([[input_data['Geography']]])

# Convert matrix to DataFrame
geo_encoded_df = pd.DataFrame(
    geo_encoded_matrix.toarray() if hasattr(geo_encoded_matrix, "toarray") else geo_encoded_matrix, 
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

c:\Avantika\Bank Churn Prediction - ANN project\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [8]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [9]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [10]:
# Combine one-hot encoded columns with input dataframe
input_df = pd.DataFrame([input_data])
input_df = pd.concat([input_df.reset_index(drop=True), geo_encoded_df], axis=1)

# Drop original 'Geography' text column as it's now encoded
input_df = input_df.drop('Geography', axis=1)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [11]:
#Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [12]:
#concatenation of onehot encoded data with input data
input_df = pd.concat([input_df.reset_index(drop=True), geo_encoded_df], axis=1)
input_df = input_df.loc[:, ~input_df.columns.duplicated()]
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [13]:

#Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [14]:
#Predict Churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step


array([[0.01019608]], dtype=float32)

In [15]:
prediction_proba = prediction[0][0]  # Assuming binary classification and you want the probability of class 1
print(prediction_proba)

0.010196083


In [17]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")   
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
